# exp068_equivalent_pixiux_inference_port inference

Inference for the exp068 full-train LightGBM boosters. The notebook generates hidden-test exp063 replay features, applies the exp068 full models, and writes `submission.csv`, prediction artifacts, diff, and SHA.

## Contents

1. Setup and configuration
2. Source artifact check
3. Exp039-style branch inference audit
4. Submission and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from exp063_branch_audit import run_exp068_full_model_inference, find_artifact_dir, FULL_MODEL_DIRNAME

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "inference.mode"))
print("Inference kernel sources:", cfg_get(config, "runtime.kaggle.inference_kernel_sources"))
print("Submission path:", paths.submission_path)


## 2. Source artifact check

In [ ]:
model_dir = find_artifact_dir(FULL_MODEL_DIRNAME, cfg_get(config, "inference.model_artifact_dir"))
print("exp068 full model dir:", model_dir)
manifest = json.loads((model_dir / "manifest.json").read_text())
print("Full model feature count:", manifest["feature_count"])
print("Full model count:", len(manifest["models"]))
display(pd.DataFrame(manifest["models"]))
print("Sample submission:", paths.sample_submission_path, "exists=", paths.sample_submission_path.exists())
display(pd.read_csv(paths.sample_submission_path, nrows=5))


## 3. Exp039-style branch inference audit

In [ ]:
summary = run_exp068_full_model_inference(
    output_dir=paths.artifacts_dir,
    submission_path=paths.submission_path,
    sample_submission_path=paths.sample_submission_path,
    model_artifact_dir=cfg_get(config, "inference.model_artifact_dir"),
    tracker_test_features_path=cfg_get(config, "data.exp063_tracker_features_test_local"),
    reference_submission_path=cfg_get(config, "inference.reference_submission_path"),
    data_dir=paths.raw_data_dir,
    feature_mode=cfg_get(config, "inference.feature_mode", "generate_exp063_replay"),
    model=cfg_get(config, "inference.model", "lgb_mean"),
    submission_target_column=cfg_get(config, "data.submission_target_column", "tvt"),
    n_jobs=int(cfg_get(config, "runtime.num_workers", 8)),
    fast=bool(cfg_get(config, "model.training.fast", False)),
    use_gpu=cfg_get(config, "inference.use_gpu", "auto"),
    max_wells=cfg_get(config, "inference.max_wells"),
)
print(json.dumps(summary, indent=2))


## 4. Submission and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "exp068_exp039_cv_full_model_inference_metrics.csv")
submission = pd.read_csv(paths.submission_path)
summary_path = paths.artifacts_dir / "exp068_exp039_cv_full_model_inference_summary.json"
diff_path = paths.artifacts_dir / "exp063_branch_submission_diff.csv"

display(metrics)
display(submission.head())
print("Submission rows:", len(submission))
print("Summary:", summary_path, "exists=", summary_path.exists())
print("Diff:", diff_path, "exists=", diff_path.exists())
print("SHA256:", summary["submission"].get("sha256"))
print("Fallback rows:", summary["submission"].get("fallback_rows"))
